# Prompt Sampling Inspector

Quick notebook to inspect prompts generated by `prompt_sampling.get_prompts`.


In [1]:
import prompt_sampling as ps
from utility import LLAMA_PROMPT_TEMPLATES

In [2]:
print("Instruction count:", len(ps.INSTRUCTIONS))
print("Adherence probe count:", len(ps.ADHERENCE_PROBES))
print(
    "Length instruction count:",
    len(
        ps.LENGTH_INSTRUCTIONS_LONG
        + ps.LENGTH_INSTRUCTIONS_MEDIUM
        + ps.LENGTH_INSTRUCTIONS_SHORT
    ),
)
print("Format instruction count:", len(ps.FORMAT_INSTRUCTIONS))
print("\nPrompt pool size:", len(ps.PROMPTS))

Instruction count: 16
Adherence probe count: 9
Length instruction count: 10
Format instruction count: 11

Prompt pool size: 495


In [3]:
def sample_prompts(
    n: int = 8,
    instruction_prob: float = 0.75,
    seed: int | None = 0,
    template_weights: tuple[float, float] = (0.25, 0.75),
    use_default_lists: bool = True,
    custom_instructions: list[str] | None = None,
    custom_prompts: list[str] | None = None,
    sampling_settings: ps.PromptSamplingSettings | None = None,
) -> list[str]:
    templates = [
        LLAMA_PROMPT_TEMPLATES["simple"],
        LLAMA_PROMPT_TEMPLATES["instruct"],
    ]

    instructions = ps.INSTRUCTIONS if use_default_lists else (custom_instructions or ps.INSTRUCTIONS)
    prompts = ps.PROMPTS if use_default_lists else (custom_prompts or ps.PROMPTS)

    sampled = ps.get_prompts(
        n_prompts=n,
        templates=templates,
        template_weights=list(template_weights),
        instructions=instructions,
        prompts=prompts,
        instruction_prob=instruction_prob,
        seed=seed,
        sampling_settings=sampling_settings,
    )

    for i, p in enumerate(sampled):
        print(f"\n--- Sample {i} ---")
        print(p)

    return sampled


In [4]:
def quick_stats(samples: list[str]) -> None:
    n = len(samples)
    has_final_answer = sum("FINAL ANSWER:" in s for s in samples)
    has_json = sum("'answer' and 'evidence'" in s for s in samples)
    one_line = sum("Output a single line only." in s for s in samples)
    avg_chars = sum(len(s) for s in samples) / max(n, 1)

    print("n:", n)
    print("avg chars/sample:", round(avg_chars, 1))
    print("contains 'FINAL ANSWER:' instruction:", has_final_answer)
    print("contains JSON instruction:", has_json)
    print("contains single-line instruction:", one_line)


In [5]:
samples = sample_prompts(n=6, instruction_prob=0.85, seed=11)
quick_stats(samples)



--- Sample 0 ---
<|image|>If details are uncertain, say what is unclear.
Use 2-3 sentences.
Wrap the final answer in double quotes.
Do not include chain-of-thought; only output the requested answer.
Characterize the likely setting and context for a dataset annotation using plain language.


--- Sample 1 ---
<|start_header_id|>user<|end_header_id|>

<|image|>Follow these instructions exactly.
If text is visible and legible, transcribe it exactly.
Return valid JSON with keys 'answer' and 'evidence'.

Task: Describe the full scene for a dataset annotation while explicitly noting uncertainty.
<|eot_id|><|start_header_id|>assistant<|end_header_id|>



--- Sample 2 ---
<|start_header_id|>user<|end_header_id|>

<|image|>Prefer concrete visual details over speculation.
Write a detailed paragraph (about 120-180 words).
Write the answer in lowercase.
Characterize the key objects and their relationships for an image retrieval index without making unsupported assumptions.
<|eot_id|><|start_header

In [6]:
# Example: stress instruction-following only
stress = sample_prompts(n=6, instruction_prob=1.0, seed=22)
quick_stats(stress)



--- Sample 0 ---
<|start_header_id|>user<|end_header_id|>

<|image|>Ground your answer in colors, shapes, or objects you can see.
Use 2-3 sentences.
Summarize the key objects and their relationships for a dataset annotation while explicitly noting uncertainty.
<|eot_id|><|start_header_id|>assistant<|end_header_id|>



--- Sample 1 ---
<|image|>Task: Caption the full scene for a visually impaired user with emphasis on observable details.

Output constraints:
If asked to classify, provide the best class and one runner-up.
Give a structured explanation with 2-3 supporting observations.


--- Sample 2 ---
<|image|>Task: Summarize the foreground and background for an image retrieval index while explicitly noting uncertainty.

Output constraints:
Ground your answer in colors, shapes, or objects you can see.
Use 2-3 sentences.


--- Sample 3 ---
<|start_header_id|>user<|end_header_id|>

<|image|>Task: Caption the main subject for an image retrieval index while explicitly noting uncertainty.
